# M4 part 3 — Feature ablation

Which feature group actually moves the needle? We trained five LSTM variants, each adding one feature group on top of the previous, and scored them on the same 70-date stratified holdout the rest of the project uses. Same architecture (encoder LSTM(64) → decoder LSTM(64) → TimeDistributed Dense), same training recipe (Huber, 60 epochs, early-stop patience 8), same residual target (`actual − TSO_fc`).

The ladder:

| # | Variant | Encoder | Decoder |
|---|---------|---------|---------|
| A | calendar only | hour/dow sin·cos | hour/dow + holiday |
| B | + load history | A + load | A |
| C | + residual lag | B + residual_hist | A |
| D | + TSO_fc decoder | C | A + tso_load_fc |
| E | + weather | D + 4× weather | D + 4× weather |

All variants train on the residual target so skill is comparable across the ladder.

In [1]:
import os
from pathlib import Path

_here = Path.cwd()
ROOT = _here if (_here / 'pyproject.toml').exists() else _here.parent
os.chdir(ROOT)

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


## 1. Results table

In [2]:
df = pd.read_csv('backtest_results/ablation_summary.csv')
df['marginal_skill'] = df['holdout_skill'].diff().fillna(df['holdout_skill'])
df[['variant', 'label', 'enc_features', 'dec_features',
    'val_implied_skill', 'holdout_skill', 'marginal_skill']]


,variant,label,enc_features,dec_features,val_implied_skill,holdout_skill,marginal_skill
0,A_calendar,calendar only,4,5,0.052023,0.047164,0.047164
1,B_load,+ load history,5,5,0.126387,0.097491,0.050326
2,C_residual,+ residual lag,6,5,0.261476,0.236834,0.139343
3,D_tso_fc,+ TSO_fc decoder,6,6,0.256178,0.228666,-0.008168
4,E_weather,+ weather,10,10,0.304077,0.242210,0.013544


## 2. Cumulative holdout skill

Each bar shows the holdout skill of the model when that feature group (and all groups to its left) are present.

In [3]:
fig = go.Figure()
fig.add_bar(
    x=df['label'], y=df['holdout_skill'],
    marker_color=['#888', '#4a90d9', '#2a8c4a', '#2a8c4a', '#cc5500'],
    text=[f'{v:+.3f}' for v in df['holdout_skill']],
    textposition='outside',
)
fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(
    title='Holdout skill vs TSO baseline (cumulative)',
    yaxis_title='Skill score (1 - MAE_model / MAE_TSO)',
    height=440, width=820, template='plotly_white',
    margin=dict(t=60, b=80),
)
fig.update_yaxes(range=[0, max(df['holdout_skill']) * 1.18])
fig.show()


## 3. Marginal contribution of each feature group

Each bar = the holdout skill **delta** when adding that feature group to the previous variant. This is what tells us where the leverage is.

In [4]:
colors = ['#888'] + [
    '#2a8c4a' if v > 0 else '#cc3030'
    for v in df['marginal_skill'].iloc[1:]
]
fig = go.Figure()
fig.add_bar(
    x=df['label'], y=df['marginal_skill'],
    marker_color=colors,
    text=[f'{v:+.3f}' for v in df['marginal_skill']],
    textposition='outside',
)
fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(
    title='Marginal skill contribution per feature group',
    yaxis_title='Δ holdout skill vs previous variant',
    height=440, width=820, template='plotly_white',
    margin=dict(t=60, b=80),
)
fig.show()


## 4. What this tells us

**Residual lag is the single biggest lever (+0.139 skill).** This is residual learning made operational: showing the model the recent `actual − TSO` error pattern in the encoder lets it learn the *systematic* bias in the TSO forecast. That's the headline insight of the project — the TSO already gets the easy 90 % right (calendar, climatology); the model only has to learn the structured remainder.

**Calendar features alone clear +0.047 skill.** Even with no load history at all, the residual target has structure the TSO underspecifies — mostly holiday-effect bias. Free skill from a 4-feature encoder.

**Load history is a smaller lever than you might guess (+0.050).** Once the residual lag is in (variant C), raw load history is largely redundant with it, but as a step on its own it adds modest level/trend information.

**Adding TSO_fc to the decoder is roughly neutral (−0.008).** This is a worthwhile *negative* result. Since the target is already `actual − TSO_fc`, exposing TSO_fc again in the decoder is information the model has already implicitly accounted for. The mild dip is noise / mild overfit on a tiny ~1000-day training set. The simpler architecture is fine — and we *checked*, which is the point.

**Weather adds +0.014 on average.** Matches notebook 06's finding: the average lift is small, but the *case-study* lift is large — on 1 May 2026 (PV-driven negative-price record) the weather model improved on the plain LSTM by **+0.51 skill** for that single day. The average masks the times weather actually matters.

### Calibration on the design choice

The residual-learning + lagged-residual feature combo isn't a decoration — it's the design that buys ~60 % of the project's skill (0.139 / 0.242 ≈ 57 %). Everything else (calendar, weather, model architecture) is incremental on top of that one decision.

## 5. What we deliberately did *not* test

- **Cross-border price lags.** 14 neighbour bidding zones live in the   parquet but aren't in the windowing pipeline yet. ~30-min plumbing   job; deferred. Likely small lift on top of weather since prices   are themselves load-driven.
- **Per-Bundesland weather.** Currently 4 nationally-aggregated   variables. Regional disaggregation could matter for renewable-heavy   zones (PV in Bayern, wind in Schleswig-Holstein) but quadruples   feature count on a small training set.
- **Different encoder architectures per variant.** All five share the   same LSTM(64) hidden size. A calendar-only model could probably   use a smaller encoder, but matching architectures keeps the   comparison apples-to-apples.